# Курсова робота
## Аналіз асортименту та попиту на товари

### Додатковий аналітичний блок

**Джерело даних:** `analysis_dataset.csv`

**Мета:** поглибити аналіз асортименту та попиту за допомогою розрахунку виручки й валового прибутку, перевірки якості даних, групування продажів, ABC/XYZ-аналізу, кореляційного аналізу та візуалізації.

Ноутбук створено на основі наданого файлу `notebook_additions.py`. Код розділено на окремі комірки, а до кожного етапу додано коротке пояснення.

## Перед початком роботи

1. Поклади `analysis_dataset.csv` у ту саму папку, що й цей ноутбук.
2. Переконайся, що встановлені бібліотеки `pandas`, `numpy`, `matplotlib` і `seaborn`.
3. Запускай комірки послідовно зверху вниз.

У першій комірці дані завантажуються з CSV, тому цей ноутбук можна запускати окремо.

## 1. Імпорт бібліотек і завантаження даних

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("analysis_dataset.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["revenue"] = df["quantity"] * df["retail_price"] * (1 - df["discount_pct"])

print("Розмір набору даних:", df.shape)
display(df.head())

Дані завантажуються з підготовленого CSV-файлу. Дата перетворюється у формат datetime, а виручка розраховується з урахуванням знижки.

## 3. Перевірка якості даних

Перевіряються пропуски, дублікати, узгодженість замовлень і потенційні викиди виручки.

In [ ]:
df["gross_profit"] = df["revenue"] - df["quantity"] * df["purchase_price"]       # валовий прибуток
df["month"] = df["date"].dt.to_period("M").astype(str)                           # місяць продажу
df["cat_type"] = df["category_name"].str.replace(r"\s*\(.*\)", "", regex=True)   # тип товару без серії та рівня
df["has_discount"] = np.where(df["discount_pct"] > 0, "Зі знижкою", "Без знижки")
df["price_segment"] = pd.cut(df["retail_price"], bins=[0, 1000, 5000, 20000, np.inf],
                             labels=["До 1 000", "1 000-5 000", "5 000-20 000", "Понад 20 000"])
print(df[["revenue", "gross_profit", "month", "cat_type", "has_discount", "price_segment"]].head())

## 4. Аналіз продажів за групами, брендами та оплатою

Порівнюються продажі, виручка, прибуток і маржа за товарними групами, брендами, способами оплати, знижками та ціновими сегментами.

In [ ]:
print(df.isna().sum()[lambda s: s > 0])
print("Повних дублікатів:", df.duplicated().sum(), "| дублів transaction_id:", df["transaction_id"].duplicated().sum())
orders = df.groupby("order_id").agg(rows=("transaction_id", "size"),
                                    dates=("date", lambda s: s.dt.date.nunique()),
                                    pays=("payment_type", lambda s: s.nunique(dropna=True)))
print("Замовлень:", len(orders), "| з >1 рядком:", (orders.rows > 1).sum(),
      "| з різними датами:", (orders.dates > 1).sum(), "| з різними способами оплати:", (orders.pays > 1).sum())
q1, q3 = df["revenue"].quantile([.25, .75]); fence = q3 + 1.5 * (q3 - q1)
print("Верхня межа IQR:", round(fence, 2), "| викидів:", (df["revenue"] > fence).sum())
print("Частка 10% найбільших рядків у виручці:", round(df["revenue"].nlargest(500).sum() / df["revenue"].sum(), 4))

## 5. Динаміка виручки за місяцями

Формуються зведені таблиці виручки за товарними групами й місяцями. Для неповного жовтня розраховується середня добова виручка.

In [ ]:
by_group = df.groupby("group").agg(lines=("transaction_id", "size"), units=("quantity", "sum"),
                                   revenue=("revenue", "sum"), profit=("gross_profit", "sum"),
                                   median_line=("revenue", "median"))
by_group["revenue_share_%"] = by_group["revenue"] / by_group["revenue"].sum() * 100
by_group["margin_%"] = by_group["profit"] / by_group["revenue"] * 100
display(by_group.sort_values("revenue", ascending=False).round(1))
display(df.groupby("brand").agg(units=("quantity", "sum"), revenue=("revenue", "sum")).sort_values("revenue", ascending=False).round(0))
display(df.groupby(df["payment_type"].fillna("Не вказано")).agg(lines=("revenue", "size"), mean=("revenue", "mean"),
                                                               median=("revenue", "median"), revenue=("revenue", "sum")).round(1))
display(df.groupby("discount_pct").agg(lines=("quantity", "size"), avg_qty=("quantity", "mean"),
                                       median_rev=("revenue", "median"),
                                       margin_pct=("gross_profit", lambda s: s.sum() / df.loc[s.index, "revenue"].sum() * 100)).round(3))
display(df.groupby("price_segment", observed=True).agg(lines=("revenue", "size"), units=("quantity", "sum"),
                                                       revenue=("revenue", "sum"), profit=("gross_profit", "sum")).round(0))

## 6. ABC-аналіз

Товари та типи товарів класифікуються за внеском у загальну виручку.

In [ ]:
days = {"2026-05": 31, "2026-06": 30, "2026-07": 31, "2026-08": 31, "2026-09": 30, "2026-10": 27}
pivot_rev = df.pivot_table(index="group", columns="month", values="revenue", aggfunc="sum")
pivot_day = pivot_rev.div(pd.Series(days), axis=1).round(0)
display(pivot_rev.round(0)); display(pivot_day)
monthly = df.groupby("month").agg(revenue=("revenue", "sum"), units=("quantity", "sum"))
monthly["revenue_per_day"] = monthly["revenue"] / pd.Series(days)
display(monthly.round(1))

## 7. XYZ-аналіз і матриця ABC/XYZ

Стабільність попиту оцінюється за коефіцієнтом варіації добових продажів; результати поєднуються з ABC-класами.

In [ ]:
def abc_class(revenue):
    s = revenue.sort_values(ascending=False); share = s / s.sum(); prev = share.cumsum() - share
    return pd.Series(np.where(prev < .80, "A", np.where(prev < .95, "B", "C")), index=s.index), share.cumsum()

prod_rev = df.groupby("product_id")["revenue"].sum()
prod_abc, _ = abc_class(prod_rev)
print("ABC за товарами:", prod_abc.value_counts().to_dict(), "| товарів зі збутом:", len(prod_rev))
print("Частка виручки 20% найкращих товарів:", round(prod_rev.nlargest(int(len(prod_rev) * .2)).sum() / prod_rev.sum(), 4))

type_rev = df.groupby("cat_type")["revenue"].sum()
type_abc, type_cum = abc_class(type_rev)
daily_units = df.pivot_table(index="cat_type", columns="month", values="quantity", aggfunc="sum", fill_value=0).div(pd.Series(days), axis=1)
cv = daily_units.std(axis=1, ddof=0) / daily_units.mean(axis=1)
xyz = pd.Series(np.where(cv <= .20, "X", np.where(cv <= .30, "Y", "Z")), index=cv.index)
matrix = pd.DataFrame({"revenue": type_rev, "cum_share_%": type_cum * 100, "ABC": type_abc, "CV": cv, "XYZ": xyz}).sort_values("revenue", ascending=False)
matrix["class"] = matrix["ABC"] + matrix["XYZ"]
display(matrix.round(2))
display(pd.crosstab(matrix["ABC"], matrix["XYZ"]))

## 8. Кореляційний аналіз Спірмена

Досліджуються монотонні зв’язки між кількістю, знижкою, цінами та виручкою, а також між ціною товару і кількістю продажів.

In [ ]:
num = ["quantity", "discount_pct", "purchase_price", "retail_price", "revenue"]
display(df[num].corr(method="spearman").round(2))
prod = df.groupby("product_id").agg(units=("quantity", "sum"), price=("retail_price", "first"))
print("Спірмен: ціна товару - кількість проданих одиниць:", round(prod.corr(method="spearman").iloc[0, 1], 3))

## 9. Boxplot виручки за групами

Діаграма розмаху дає змогу порівняти розподіл виручки між групами товарів.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
order = by_group.sort_values("revenue", ascending=False).index
plt.figure(figsize=(10, 5.5))
sns.boxplot(data=df, x="group", y="revenue", order=order, showfliers=True)
plt.yscale("log"); plt.title("Розподіл виручки за записом за групами товарів")
plt.xlabel("Група товарів"); plt.ylabel("Виручка за записом, грн (логарифмічна шкала)")
plt.tight_layout(); plt.show()

s = prod_rev.sort_values(ascending=False)
x = np.arange(1, len(s) + 1) / len(s) * 100; y = s.cumsum() / s.sum() * 100
plt.figure(figsize=(9, 5)); plt.plot(x, y, label="Фактична крива")
plt.plot([0, 100], [0, 100], "--", color="gray", label="Рівномірний розподіл")
plt.axhline(80, color="red", lw=.8); plt.title("Крива Парето для виручки за товарами")
plt.xlabel("Частка товарів, % (за спаданням виручки)"); plt.ylabel("Накопичена частка виручки, %"); plt.legend()
plt.tight_layout(); plt.show()

## Висновок

У роботі виконано поглиблений аналіз асортименту та попиту на товари. Розраховано виручку з урахуванням знижок і валовий прибуток, сформовано додаткові аналітичні ознаки, перевірено якість даних та узгодженість записів у межах замовлень.

Проведено групування продажів за товарними групами, брендами, способами оплати, знижками й ціновими сегментами. Проаналізовано місячну виручку, виконано ABC/XYZ-аналіз, побудовано матрицю класифікації, досліджено кореляції методом Спірмена та створено графіки boxplot і Парето.

Конкретні числові результати, лідерів асортименту та остаточні висновки потрібно визначити після запуску комірок на фактичному файлі `analysis_dataset.csv`.

## Примітки

- Ноутбук зберігає розрахунки та назви полів із наданого Python-файлу.
- Потрібні стовпці: `date`, `quantity`, `retail_price`, `discount_pct`, `purchase_price`, `category_name`, `transaction_id`, `order_id`, `payment_type`, `group`, `brand`, `product_id`.
- У вихідному коді для частки найбільших рядків залишено `nlargest(500)` без зміни.
- Кількість днів у місяцях задана у вихідному коді: травень–вересень 2026 року та жовтень до 27.10.2026.